In [8]:
import torch
from transformers import AutoModelForCausalLM,AutoTokenizer,BitsAndBytesConfig
from peft import PeftModel

In [9]:
BASE_PATH = "/home/ayush/time_series_llm/models/base_model"
ADAPTER_PATH = "/home/ayush/time_series_llm/models/Mod_ver_1_syn_1/checkpoint-225"
SAVE_PATH = "/home/ayush/time_series_llm/models/merged_Mod_1_ver_1_qwen2"


In [10]:
tokenizer = AutoTokenizer.from_pretrained(
    BASE_PATH,
    trust_remote_code = True)

In [11]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [12]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",          # best for LLMs
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

In [13]:
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_PATH,
    quantization_config = bnb_config, 
    device_map={"":0},           
    trust_remote_code=True,
)


Loading checkpoint shards: 100%|██████████| 2/2 [00:29<00:00, 14.66s/it]


In [14]:
model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH,
    is_trainable=False
)

In [15]:
merged_model = model.merge_and_unload()

/home/ayush/time_series_llm/llm_env/lib/python3.12/site-packages/peft/tuners/lora/bnb.py:373: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


In [16]:
merged_model.save_pretrained(
    SAVE_PATH,
    safe_serialization=True
)

tokenizer.save_pretrained(SAVE_PATH)

('/home/ayush/time_series_llm/models/merged_Mod_1_ver_1_qwen2/tokenizer_config.json',
 '/home/ayush/time_series_llm/models/merged_Mod_1_ver_1_qwen2/special_tokens_map.json',
 '/home/ayush/time_series_llm/models/merged_Mod_1_ver_1_qwen2/vocab.json',
 '/home/ayush/time_series_llm/models/merged_Mod_1_ver_1_qwen2/merges.txt',
 '/home/ayush/time_series_llm/models/merged_Mod_1_ver_1_qwen2/added_tokens.json',
 '/home/ayush/time_series_llm/models/merged_Mod_1_ver_1_qwen2/tokenizer.json')

In [ ]:


def generate(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=500,
            do_sample=False,
            use_cache=True,
            eos_token_id=tokenizer.eos_token_id
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)
print(generate(prompt))

NameError: name 'tokenizer' is not defined